# Hyperparameter Fine-Tuning: Random Forest & MLP
Train on the augmented training set, evaluate each combination on the held-out test set.

In [27]:
from pathlib import Path
import os
import numpy as np
import cv2
from tqdm.auto import tqdm

IMAGES_PATH = Path("../resources/cloud-images/cloud-classifier-1")

def index_labeled_images(images_path=IMAGES_PATH):
    cloud_labels = []
    images_path = Path(images_path)
    labeled_images = {}
    if not images_path.exists():
        return labeled_images, cloud_labels
    for cloud_dir in sorted(p for p in images_path.iterdir() if p.is_dir()):
        cloud_labels.append(cloud_dir.name)
        for img_path in sorted(p for p in cloud_dir.iterdir() if p.is_file()):
            labeled_images[img_path.name] = {"label": cloud_dir.name, "path": str(img_path)}
    return labeled_images, cloud_labels

def extract_labels(labeled_images):
    images, labels = [], []
    for image_name in tqdm(labeled_images, total=len(labeled_images)):
        path = labeled_images[image_name]['path']
        if not os.path.exists(path):
            continue
        image = cv2.imread(path)
        if image is None:
            continue
        image = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        image = cv2.resize(image, (128, 128))
        images.append(image)
        labels.append(labeled_images[image_name]['label'])
    return np.array(images), np.array(labels)

labeled_images, cloud_labels = index_labeled_images()
images, labels = extract_labels(labeled_images)
print(f"Loaded {len(images)} images, classes: {cloud_labels}")

100%|██████████| 698/698 [00:00<00:00, 1076.39it/s]

Loaded 698 images, classes: ['clear_sky', 'cloud']


In [28]:
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.preprocessing import LabelEncoder

ordinal_encoder = LabelEncoder()
labels_encoded = ordinal_encoder.fit_transform(labels)

sss = StratifiedShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(sss.split(images, labels_encoded))

X_train, X_test = images[train_idx], images[test_idx]
y_train, y_test = labels_encoded[train_idx], labels_encoded[test_idx]

print("train:", X_train.shape, y_train.shape)
print("test: ", X_test.shape, y_test.shape)

train: (558, 128, 128) (558,)
test:  (140, 128, 128) (140,)


In [29]:
import torch
import torchvision.transforms.v2 as T
from torchvision.transforms.v2 import InterpolationMode

IMG_SIZE = (128, 128)
MAX_FRAC = 0.14

border_translation = T.RandomAffine(
    degrees=0, translate=(MAX_FRAC, MAX_FRAC),
    interpolation=InterpolationMode.BILINEAR, fill=0
)
wrap_translation = T.Lambda(lambda x: torch.roll(
    x,
    shifts=(
        int(torch.randint(-int(MAX_FRAC * x.shape[-2]), int(MAX_FRAC * x.shape[-2]) + 1, (1,)).item()),
        int(torch.randint(-int(MAX_FRAC * x.shape[-1]), int(MAX_FRAC * x.shape[-1]) + 1, (1,)).item()),
    ),
    dims=(-2, -1),
))

stacked = T.Compose([
    T.RandomChoice([wrap_translation, border_translation]),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.15),
    T.RandomAffine(degrees=20, scale=(0.90, 1.10), interpolation=InterpolationMode.BILINEAR, fill=0),
    T.RandomResizedCrop(size=IMG_SIZE, scale=(0.80, 1.00), ratio=(0.90, 1.10), interpolation=InterpolationMode.BILINEAR),
])
one_of = T.RandomChoice([
    wrap_translation, border_translation,
    T.RandomChoice([T.RandomHorizontalFlip(p=1.0), T.RandomVerticalFlip(p=1.0)]),
    T.RandomRotation(degrees=20, interpolation=InterpolationMode.BILINEAR, fill=0),
])

augmentation_transform = T.Compose([
    T.Resize(IMG_SIZE),
    T.ToImage(),
    T.ToDtype(torch.float32, scale=True),
    T.RandomChoice([stacked, one_of]),
])

In [30]:
target_count_per_class = 2400

X_aug_list, y_aug_list = list(X_train), list(y_train)

for class_idx in range(len(ordinal_encoder.classes_)):
    cls_imgs = X_train[y_train == class_idx]
    multiple = target_count_per_class // len(cls_imgs)
    print(f"{ordinal_encoder.classes_[class_idx]}: {len(cls_imgs)} images, augmenting x{multiple - 1}")
    for img in cls_imgs:
        img_torch = torch.from_numpy(img)
        for _ in range(multiple - 1):
            aug = augmentation_transform(img_torch.unsqueeze(0)).squeeze(0).numpy()
            X_aug_list.append(aug)
            y_aug_list.append(class_idx)

X_train_aug = np.array(X_aug_list)
y_train_aug = np.array(y_aug_list)

idx = np.random.permutation(len(X_train_aug))
X_train_aug, y_train_aug = X_train_aug[idx], y_train_aug[idx]

print(f"\nAugmented training set: {X_train_aug.shape}")

clear_sky: 279 images, augmenting x7
cloud: 279 images, augmenting x7

Augmented training set: (4464, 128, 128)


In [31]:
X_train_flat = X_train_aug.reshape(len(X_train_aug), -1).astype(np.float32) / 255.0
X_test_flat  = X_test.reshape(len(X_test), -1).astype(np.float32) / 255.0

print("train flat:", X_train_flat.shape)
print("test flat: ", X_test_flat.shape)

train flat: (4464, 16384)
test flat:  (140, 16384)


## Random Forest — Hyperparameter Search

In [34]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import ParameterSampler
from sklearn.metrics import f1_score
from scipy.stats import randint

rf_param_dist = {
    "n_estimators":          randint(50, 150),
    "max_depth":             [5, 10, 15, 20],
    "min_samples_split":     randint(2, 15),
    "min_samples_leaf":      randint(1, 8),
    "max_features":          ["sqrt", "log2"],
    "criterion":             ["gini", "entropy"],
    "bootstrap":             [True, False],
    "min_impurity_decrease": [0.0, 0.01, 0.05],
}

n_iter = 30
sampler = ParameterSampler(rf_param_dist, n_iter=n_iter, random_state=42)

rf_results = []
for params in tqdm(sampler, total=n_iter):
    clf = RandomForestClassifier(random_state=42, n_jobs=-1, **params)
    clf.fit(X_train_flat, y_train_aug)
    y_pred = clf.predict(X_test_flat)
    rf_results.append({**params, "f1_macro": f1_score(y_test, y_pred, average="macro")})

rf_results_df = pd.DataFrame(rf_results).sort_values("f1_macro", ascending=False)
print(rf_results_df.head(10))

100%|██████████| 30/30 [00:23<00:00,  1.29it/s]

    bootstrap criterion  max_depth max_features  min_impurity_decrease  \
14      False      gini         20         log2                    0.0   
16      False      gini         15         sqrt                    0.0   
29       True      gini         10         sqrt                    0.0   
27       True      gini         10         sqrt                    0.0   
18       True   entropy         20         log2                    0.0   
1        True      gini         15         log2                    0.0   
7        True      gini         10         log2                    0.0   
6       False   entropy          5         sqrt                    0.0   
15       True      gini          5         sqrt                    0.0   
25      False   entropy          5         log2                    0.0   

    min_samples_leaf  min_samples_split  n_estimators  f1_macro  
14                 1                 14            64  0.928513  
16                 3                  4            82

In [35]:
from sklearn.metrics import classification_report, accuracy_score

best_params = rf_results_df.iloc[0].drop("f1_macro").to_dict()
for k in ["n_estimators", "min_samples_split", "min_samples_leaf"]:
    best_params[k] = int(best_params[k])

print("Best params:", best_params)

best_rf = RandomForestClassifier(random_state=42, n_jobs=-1, **best_params)
best_rf.fit(X_train_flat, y_train_aug)
y_pred = best_rf.predict(X_test_flat)

print(f"\nAccuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 macro:  {f1_score(y_test, y_pred, average='macro'):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=ordinal_encoder.classes_))

Best params: {'bootstrap': False, 'criterion': 'gini', 'max_depth': 20, 'max_features': 'log2', 'min_impurity_decrease': 0.0, 'min_samples_leaf': 1, 'min_samples_split': 14, 'n_estimators': 64}

Accuracy:  0.9286
F1 macro:  0.9285

              precision    recall  f1-score   support

   clear_sky       0.95      0.90      0.93        70
       cloud       0.91      0.96      0.93        70

    accuracy                           0.93       140
   macro avg       0.93      0.93      0.93       140
weighted avg       0.93      0.93      0.93       140



## MLP — Hyperparameter Search

In [36]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from scipy.stats import loguniform

mlp_param_dist = {
    "hidden_layer_sizes": [(64,), (128,), (256,), (64, 32), (128, 64), (256, 128), (128, 64, 32)],
    "activation":         ["relu", "tanh"],
    "alpha":              loguniform(1e-5, 1e-1),
    "learning_rate_init": loguniform(1e-4, 1e-2),
    "batch_size":         [32, 64, 128, 256],
    "solver":             ["adam"],
}

n_iter = 20
sampler = ParameterSampler(mlp_param_dist, n_iter=n_iter, random_state=42)

mlp_results = []
for params in tqdm(sampler, total=n_iter):
    pipe = Pipeline([
        ("scaler", StandardScaler()),
        ("clf", MLPClassifier(random_state=42, max_iter=300, **params))
    ])
    pipe.fit(X_train_flat, y_train_aug)
    y_pred = pipe.predict(X_test_flat)
    mlp_results.append({**params, "f1_macro": f1_score(y_test, y_pred, average="macro")})

mlp_results_df = pd.DataFrame(mlp_results).sort_values("f1_macro", ascending=False)
print(mlp_results_df.head(10))

  0%|          | 0/20 [00:00<?, ?it/s]/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-p

   activation     alpha  batch_size hidden_layer_sizes  learning_rate_init  \
16       relu  0.001212          64          (128, 64)            0.000599   
1        relu  0.000042         128      (128, 64, 32)            0.000131   
15       tanh  0.050890         128      (128, 64, 32)            0.000150   
2        tanh  0.000216         256             (256,)            0.000110   
0        relu  0.015352         128             (256,)            0.003626   
3        tanh  0.007727          64         (256, 128)            0.000100   
12       relu  0.043379         256             (128,)            0.002114   
9        relu  0.000011          32           (64, 32)            0.001338   
8        relu  0.000015         128      (128, 64, 32)            0.000219   
14       relu  0.075568          64           (64, 32)            0.007568   

   solver  f1_macro  
16   adam  0.804257  
1    adam  0.799632  
15   adam  0.755180  
2    adam  0.753718  
0    adam  0.727179  
3    adam

In [37]:
best_mlp_params = mlp_results_df.iloc[0].drop("f1_macro").to_dict()
print("Best params:", best_mlp_params)

best_mlp = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", MLPClassifier(random_state=42, max_iter=300, **best_mlp_params))
])
best_mlp.fit(X_train_flat, y_train_aug)
y_pred = best_mlp.predict(X_test_flat)

print(f"\nAccuracy:  {accuracy_score(y_test, y_pred):.4f}")
print(f"F1 macro:  {f1_score(y_test, y_pred, average='macro'):.4f}")
print()
print(classification_report(y_test, y_pred, target_names=ordinal_encoder.classes_))

Best params: {'activation': 'relu', 'alpha': 0.0012115379992469583, 'batch_size': 64, 'hidden_layer_sizes': (128, 64), 'learning_rate_init': 0.0005989003672254305, 'solver': 'adam'}


/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b



Accuracy:  0.6143
F1 macro:  0.5870

              precision    recall  f1-score   support

   clear_sky       0.74      0.36      0.48        70
       cloud       0.58      0.87      0.69        70

    accuracy                           0.61       140
   macro avg       0.66      0.61      0.59       140
weighted avg       0.66      0.61      0.59       140



/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/lucasnseyep/code/LucasNseyep/belle-weder/.venv/lib/python3.13/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b
